# Лабораторная работа 3.3
## Уравнение теплопроводности на полубесконечной прямой
### Метод размерностей и подобия

$$\frac{\partial U}{\partial t} = a^2 \frac{\partial^2 U}{\partial x^2}, \quad x > 0, \; t > 0$$

Н.У.: $U(x, 0) = T_0$. Г.У.: $U(0, t) = 0$.

**Метод 1** (аналитика): $\;U(x,t) = T_0 \cdot \operatorname{erf}\!\left(\dfrac{x}{2a\sqrt{t}}\right)$

**Метод 2** (инт. Пуассона): $\;U(x,t) = \dfrac{T_0}{2a\sqrt{\pi t}} \displaystyle\int_0^{\infty} \!\left[e^{-\frac{(x-z)^2}{4a^2 t}} - e^{-\frac{(x+z)^2}{4a^2 t}}\right] dz$

**Метод 3** (неявная схема): $\;-r\,U_{i-1}^{m+1} + (1+2r)\,U_i^{m+1} - r\,U_{i+1}^{m+1} = U_i^{m}, \quad r = \dfrac{a^2\tau}{h^2}$

In [26]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.special import erf
from scipy.integrate import quad
from IPython.display import HTML

a = 1.0
T0 = 100.0
T_end = 60.0
L = 3 * a * np.sqrt(T_end) * 1.2

dx = 0.1
dt = 0.01
Nx = int(L / dx) + 1
Nt = int(T_end / dt) + 1
x = np.linspace(0, L, Nx)
t = np.linspace(0, T_end, Nt)
dt = t[1] - t[0]

In [27]:
def solve_analytical(x, t_val):
    if t_val <= 0:
        u = np.full_like(x, T0)
        u[0] = 0.0
        return u
    return T0 * erf(x / (2 * a * np.sqrt(t_val)))


def solve_poisson(x_arr, t_val):
    if t_val <= 0:
        u = np.full_like(x_arr, T0)
        u[0] = 0.0
        return u
    coeff = T0 / (2 * a * np.sqrt(np.pi * t_val))
    d = 4 * a**2 * t_val
    sigma = np.sqrt(d / 2)
    result = np.zeros_like(x_arr)
    for i, xi in enumerate(x_arr):
        if xi == 0:
            continue
        upper = xi + 6 * sigma
        f = lambda z: np.exp(-(xi - z)**2 / d) - np.exp(-(xi + z)**2 / d)
        result[i] = coeff * quad(f, 0, upper)[0]
    return result

In [28]:
def solve_tridiag(lo, diag, up, rhs):
    n = len(rhs)
    for i in range(1, n):
        w = lo[i] / diag[i - 1]
        diag[i] -= w * up[i - 1]
        rhs[i] -= w * rhs[i - 1]
    x = np.zeros(n)
    x[-1] = rhs[-1] / diag[-1]
    for i in range(n - 2, -1, -1):
        x[i] = (rhs[i] - up[i] * x[i + 1]) / diag[i]
    return x


def solve_implicit(x, t):
    Nx, Nt = len(x), len(t)
    h = x[1] - x[0]
    tau = t[1] - t[0]
    r = a**2 * tau / h**2

    U = np.zeros((Nx, Nt))
    U[:, 0] = T0
    U[0, 0] = 0.0

    for j in range(Nt - 1):
        lo = np.full(Nx, -r)
        diag = np.full(Nx, 1 + 2 * r)
        up = np.full(Nx, -r)
        rhs = U[:, j].copy()

        lo[0], diag[0], up[0], rhs[0] = 0, 1, 0, 0
        lo[-1], diag[-1], up[-1], rhs[-1] = 0, 1, 0, T0

        U[:, j + 1] = solve_tridiag(lo, diag, up, rhs)

    return U

In [29]:
U_num = solve_implicit(x, t)

dt_frame = 0.1
frame_times = np.arange(0, T_end + dt_frame / 2, dt_frame)
frame_idx = np.array([np.argmin(np.abs(t - tt)) for tt in frame_times])
n_frames = len(frame_idx)

x_lim = L
x_plot = x
x_po = x_plot[::1]

U_po_cache = {}
for fi in frame_idx:
    U_po_cache[fi] = solve_poisson(x_po, t[fi])

In [ ]:
plt.rcParams['animation.embed_limit'] = 30

fig, ax = plt.subplots(figsize=(10, 5))
ln_num, = ax.plot([], [], 'b-',  lw=2,   label='Сеточный')
ln_an,  = ax.plot([], [], 'g-.', lw=2,   label='Размерностей')
ln_po,  = ax.plot([], [], 'r--', lw=1.5, label='Инт. Пуассона')
txt = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=12,
              verticalalignment='top',
              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlim(0, x_lim)
ax.set_ylim(-5, T0 + 5)
ax.set_xlabel('x')
ax.set_ylabel('U(x, t)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)


def animate(frame):
    j = frame_idx[frame]
    tv = t[j]
    ln_num.set_data(x_plot, U_num[:, j])
    ln_an.set_data(x_plot, solve_analytical(x_plot, tv))
    ln_po.set_data(x_po, U_po_cache[j])
    txt.set_text(f't = {tv:.2f}')
    return ln_num, ln_an, ln_po, txt


anim = FuncAnimation(fig, animate, frames=n_frames,
                     interval=80, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

<IPython.core.display.HTML object>